In [0]:
# ! pip install holidays

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date
import holidays

In [0]:
from pyspark import SparkFiles
p1 ="https://github.com/erkansirin78/datasets/raw/master/Store_Item_Demand_Forecasting/train.csv"
sc.addFile(p1)
df = spark.read.csv("file:///"+SparkFiles.get("train.csv"), header=True, inferSchema=True).withColumnRenamed("sales", "label")
     

In [0]:

display(df.limit(5))

date,store,item,label
2013-01-01,1,1,13
2013-01-02,1,1,11
2013-01-03,1,1,14
2013-01-04,1,1,13
2013-01-05,1,1,10


In [0]:
df_item_12 = df.filter("item = 12")

In [0]:
from pyspark.ml import Transformer
from pyspark.sql.functions import lit, udf
from pyspark.ml.param.shared import HasInputCols, HasOutputCol
from datetime import date

@udf(IntegerType())
def is_holiday(date_str: date, country_code: str='TR'):
    date_str = str(date_str)
    country_holidays = holidays.CountryHoliday(country_code) 
    date_obj = date.fromisoformat(date_str)
    if date_obj in country_holidays:
        return 1
    else:
        return 0
        
class AddDateFeaturesTransformer(Transformer, HasInputCols, HasOutputCol):
    def __init__(self, inputCol=None, outputCols=None, date=None, country_code=None):
        super(AddDateFeaturesTransformer, self).__init__()
        self.inputCol = inputCol
        self.outputCols = outputCols
        self.date = date
        self.country_code = country_code
    
    def _transform(self, df):
        df = df.withColumn(self.outputCols[0], F.year(self.inputCol)) \
        .withColumn(self.outputCols[1], F.month(self.inputCol)) \
        .withColumn(self.outputCols[2], F.dayofweek(self.inputCol)) \
        .withColumn(self.outputCols[3], is_holiday(self.inputCol))

        return df

In [0]:
sct = AddDateFeaturesTransformer(inputCol='date', outputCols=['y','m','d','is_holiday'])

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(['y','m','d','is_holiday']) \
.setOutputCol('features')

In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(numTrees=100)

In [0]:
from pyspark.ml.pipeline import Pipeline

pipeline_obj = Pipeline().setStages([sct, assembler, rf])

In [0]:
train_df, test_df = df_item_12.randomSplit([0.7, 0.3], seed=142)

In [0]:
pipeline_model = pipeline_obj.fit(train_df)

In [0]:
display(test_df.limit(5))

date,store,item,label
2013-01-01,2,12,55
2013-01-01,4,12,36
2013-01-01,7,12,26
2013-01-01,9,12,35
2013-01-02,1,12,35


In [0]:
transformed_df = pipeline_model.transform(test_df)

In [0]:
display(transformed_df.limit(10))

date,store,item,label,y,m,d,is_holiday,features,prediction
2013-01-01,2,12,55,2013,1,3,1,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 3.0, 1.0))",39.227749376615044
2013-01-01,4,12,36,2013,1,3,1,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 3.0, 1.0))",39.227749376615044
2013-01-01,7,12,26,2013,1,3,1,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 3.0, 1.0))",39.227749376615044
2013-01-01,9,12,35,2013,1,3,1,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 3.0, 1.0))",39.227749376615044
2013-01-02,1,12,35,2013,1,4,0,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 4.0, 0.0))",39.80984074357983
2013-01-02,5,12,18,2013,1,4,0,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 4.0, 0.0))",39.80984074357983
2013-01-03,5,12,32,2013,1,5,0,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 5.0, 0.0))",42.8762855739726
2013-01-03,9,12,38,2013,1,5,0,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 5.0, 0.0))",42.8762855739726
2013-01-04,1,12,45,2013,1,6,0,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 6.0, 0.0))",43.97062195055592
2013-01-04,2,12,51,2013,1,6,0,"Map(vectorType -> dense, length -> 4, values -> List(2013.0, 1.0, 6.0, 0.0))",43.97062195055592


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:

evaluator = RegressionEvaluator(metricName='r2')

In [0]:
r2 = evaluator.evaluate(transformed_df)

In [0]:
r2

Out[38]: 0.42600757504394127